In [1]:
import pickle

with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

KeyError: 'bio12'

In [2]:
import rasterio
from tqdm import tqdm

with rasterio.open(f"data/CHELSA_bio01_1981-2010_V.2.1.tif") as src:
    for entry in tqdm(metadata):
        lon = entry["center_lon"]
        lat = entry["center_lat"]
        val = list(src.sample([(lon, lat)]))[0][0]
        entry["bio01"] = val * src.scales[0] + src.offsets[0]

with rasterio.open(f"data/CHELSA_bio04_1981-2010_V.2.1.tif") as src:
    for entry in tqdm(metadata):
        lon = entry["center_lon"]
        lat = entry["center_lat"]
        val = list(src.sample([(lon, lat)]))[0][0]
        entry["bio04"] = (val * src.scales[0] + src.offsets[0]) / 100.0

with rasterio.open(f"data/CHELSA_bio12_1981-2010_V.2.1.tif") as src:
    for entry in tqdm(metadata):
        lon = entry["center_lon"]
        lat = entry["center_lat"]
        val = list(src.sample([(lon, lat)]))[0][0]
        entry["bio12"] = val * src.scales[0] + src.offsets[0]

with rasterio.open(f"data/CHELSA_bio15_1981-2010_V.2.1.tif") as src:
    for entry in tqdm(metadata):
        lon = entry["center_lon"]
        lat = entry["center_lat"]
        val = list(src.sample([(lon, lat)]))[0][0]
        entry["bio15"] = val * src.scales[0] + src.offsets[0]

with open("data/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)




100%|██████████| 89088/89088 [00:26<00:00, 3312.64it/s]


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("data/results.csv")

# Change regressor field:
# - "rff" if n_components is not None
# - "rf" otherwise
df["regressor"] = df["rff_component_count"].notna().map({True: "rff", False: "rf"})

# Set capacity:
# - n_components value if available
# - keep existing capacity otherwise
df["capacity"] = df["rff_component_count"].combine_first(df["capacity"])

# Delete n_components column
df = df.drop(columns=["rff_component_count"])

df["capacity"] = df["capacity"].astype(int)
print(df[df["regressor"] == "rf"].head())


# Save result
df.to_csv("data/results_.csv", index=False)

     fold          model_name target  layer modality_group  train_rmse  \
100     0  terramind_v1_large  bio01     23            all    0.898921   
101     1  terramind_v1_large  bio01     23            all    0.891797   
102     2  terramind_v1_large  bio01     23            all    0.902538   
103     3  terramind_v1_large  bio01     23            all    0.899495   
104     4  terramind_v1_large  bio01     23            all    0.904103   

     val_rmse  train_rmse_unscaled  val_rmse_unscaled  train_r2    val_r2  \
100  0.799551            10.517214           9.354602  0.191940  0.206465   
101  0.723716            10.668471           8.657737  0.204697  0.133839   
102  1.000927            10.156360          11.263541  0.185426  0.118497   
103  0.975707            10.165525          11.026825  0.190909  0.167246   
104  1.056431            10.061135          11.756281  0.182597  0.171819   

    regressor  capacity  
100        rf         1  
101        rf         1  
102        rf 